# OptimalBinning 统一接口示例

本示例展示 hscredit 中 `OptimalBinning` 统一接口的使用方法。所有分箱方法都整合到这一个类中,通过 `method` 参数选择不同的分箱算法。

## 支持的分箱方法

| 方法 | 说明 | 特点 |
|------|------|------|
| `uniform` | 等宽分箱 | 简单快速,将数值范围等分 |
| `quantile` | 等频分箱 | 每箱样本数相等 |
| `tree` | 决策树分箱 | 基于信息增益选择切分点 |
| `chi_merge` | 卡方分箱 | 基于卡方统计量合并 |
| `optimal_ks` | 最优KS分箱 | 最大化KS统计量 |
| `optimal_iv` | 最优IV分箱 | 最大化IV值(推荐) |
| `mdlp` | MDLP分箱 | 信息论,自动确定分箱数 |
| `cart` | CART分箱 | 参考optbinning实现 |
| `monotonic` | 单调性约束分箱 | 支持U型/倒U型/凸/凹 |
| `genetic` | 遗传算法分箱 | 全局优化 |
| `smooth` | 平滑分箱 | 正则化 |
| `kernel_density` | 核密度分箱 | 密度估计 |
| `best_lift` | Best Lift分箱 | 提升度优化 |
| `target_bad_rate` | 目标坏样本率分箱 | 指定坏样本率阈值 |
| `kmeans` | K-Means分箱 | 聚类算法分箱 |
| `optimal` | 最优分箱(向后兼容) | 等同于OptimalBinning类 |

## 1. 环境准备

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# 添加项目路径
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from hscredit.core.binning import OptimalBinning

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✅ OptimalBinning 导入成功")

✅ OptimalBinning 导入成功


## 2. 加载数据

In [2]:
# 加载示例数据
data_path = Path("hscredit.xlsx")
df = pd.read_excel(data_path)

print(f"数据集形状: {df.shape}")
print(f"\n列名: {df.columns.tolist()}")
print(f"\n数据类型:\n{df.dtypes}")

数据集形状: (22729, 12)

列名: ['MOB1', 'MOB2', '青云24', '游昆定制分80', '百融定制分V9', '中智小牛分C3', '度小满欺诈因子V6PRO多头版', '百行百川分FPV1', '游昆设备黑名单', '业务类型', '流量渠道', '放款期数']

数据类型:
MOB1                 int64
MOB2                 int64
青云24               float64
游昆定制分80            float64
百融定制分V9            float64
中智小牛分C3            float64
度小满欺诈因子V6PRO多头版    float64
百行百川分FPV1          float64
游昆设备黑名单            float64
业务类型                object
流量渠道                object
放款期数                 int64
dtype: object


In [3]:
# 选择数值型特征作为示例
# 使用 MOB1 > 0 作为目标变量(0/1分类,MOB1>0表示坏样本)
numeric_features = [
    '青云24', '游昆定制分80', '百融定制分V9', '中智小牛分C3',
    '度小满欺诈因子V6PRO多头版', '百行百川分FPV1', '游昆设备黑名单'
]

X = df[numeric_features].copy()
# MOB1 > 0 作为坏样本(二分类)
y = (df['MOB1'] > 0).astype(int)

# 处理缺失值(简单填充)
X = X.fillna(X.median())

print(f"特征数据: {X.shape}")
print(f"\n目标变量分布:\n{y.value_counts()}")
print(f"\n坏样本率: {y.mean():.2%}")

print(f"\n特征描述统计:")
X.describe()

特征数据: (22729, 7)

目标变量分布:
MOB1
0    19344
1     3385
Name: count, dtype: int64

坏样本率: 14.89%

特征描述统计:


,青云24,游昆定制分80,百融定制分V9,中智小牛分C3,度小满欺诈因子V6PRO多头版,百行百川分FPV1,游昆设备黑名单
count,22729.000000,22729.000000,22729.000000,22729.000000,22729.000000,22729.000000,22729.000000
mean,600.564477,707.860883,757.385559,615.119671,50.604678,225.452726,1.893836
std,65.781741,13.494159,89.727765,96.118537,3.118849,47.812725,1.342140
min,448.000000,627.000000,468.149994,300.000000,30.799999,1.000000,0.000000
25%,554.000000,701.000000,696.109985,550.000000,50.590000,228.000000,0.000000
50%,600.000000,714.000000,765.719971,612.000000,50.590000,228.000000,2.000000
75%,646.000000,718.000000,824.070007,677.000000,50.590000,228.000000,3.000000
max,850.000000,718.000000,990.340027,850.000000,67.430000,456.000000,4.000000


## 3. 基础使用

In [4]:
# 3.1 使用最优IV分箱(最常用)
iv_binner = OptimalBinning(method='optimal_iv', max_n_bins=5)
iv_binner.fit(X, y)

# 查看某个特征的分箱结果
feature = '青云24'
bin_table = iv_binner.get_bin_table(feature)

print(f"【{feature}】最优IV分箱结果:")
print(f"IV值: {bin_table['分档IV值'].sum():.4f}")
print(f"\n分箱详情:")
display_cols = ['分箱', '样本总数', '样本占比', '坏样本率', '分档WOE值', '分档IV值']
bin_table[display_cols]

【青云24】最优IV分箱结果:
IV值: 0.0819

分箱详情:


,分箱,样本总数,样本占比,坏样本率,分档WOE值,分档IV值
0,0,5535,0.243521,0.191328,0.301624,0.024571
1,1,10050,0.442166,0.155423,0.050342,0.001141
2,2,357,0.015707,0.084034,-0.645734,0.005193
3,3,4931,0.216948,0.121679,-0.233596,0.010895
4,4,1856,0.081658,0.072198,-0.810373,0.040060


In [5]:
# 查看切分点
print("各特征的切分点:")
print("="*60)
for feature in X.columns:
    splits = iv_binner.splits_.get(feature, [])
    print(f"{feature:20s}: {splits}")

各特征的切分点:
青云24                : [552.69 632.11 635.72 693.48]
游昆定制分80             : [697.  699.5 703.5 716. ]
百融定制分V9             : [658.5298 761.5483 807.2921 847.2135]
中智小牛分C3             : [541.   674.9  752.15 757.3 ]
度小满欺诈因子V6PRO多头版     : [43.2287 50.0856 52.8292 56.9395]
百行百川分FPV1           : [149.04 245.76 271.8  372.24]
游昆设备黑名单             : []


## 4. 多种方法对比

In [6]:
# 对比不同方法对同一特征的分箱效果
methods = ['uniform', 'quantile', 'tree', 'chi_merge', 'optimal_ks', 'optimal_iv', 
           'mdlp', 'cart', 'monotonic', 'genetic', 'smooth', 'kernel_density', 
           'best_lift', 'target_bad_rate', 'kmeans']
feature = '青云24'

print(f"【{feature}】不同分箱方法对比:")
print("="*70)
print(f"{'方法':<15} {'分箱数':>8} {'IV值':>10} {'单调性':>10}")
print("-"*70)

results = []
for method in methods:
    binner = OptimalBinning(method=method, max_n_bins=5)
    binner.fit(X[[feature]], y)
    bin_table = binner.get_bin_table(feature)
    n_bins = len([b for b in bin_table['分箱'] if b != 'missing'])
    iv = bin_table['分档IV值'].sum()
    
    # 检查单调性
    bad_rates = bin_table[bin_table['分箱'] != 'missing']['坏样本率'].values
    is_monotonic = all(np.diff(bad_rates) <= 0.001) or all(np.diff(bad_rates) >= -0.001)
    
    results.append({'method': method, 'n_bins': n_bins, 'iv': iv, 'monotonic': is_monotonic})
    print(f"{method:<15} {n_bins:>8} {iv:>10.4f} {'✓' if is_monotonic else '':>10}")

【青云24】不同分箱方法对比:
方法                   分箱数        IV值        单调性
----------------------------------------------------------------------
uniform                5     0.0626          ✓
quantile               5     0.0593          ✓
tree                   5     0.0830          ✓
chi_merge             10     0.0726           
optimal_ks             5     0.0487           
optimal_iv             5     0.0819           
mdlp                   4     0.0763          ✓
cart                   5     0.0830          ✓
monotonic              5     0.0740          ✓
genetic                4     0.0723          ✓
smooth                 2     0.0517          ✓
kernel_density         2     0.0382          ✓
best_lift              4     0.0746          ✓
target_bad_rate        5     0.0438          ✓
kmeans                 5     0.0717          ✓


## 4.2 高级分箱方法示例

In [7]:
# 遗传算法分箱 - 全局优化
genetic_binner = OptimalBinning(method='genetic', max_n_bins=5)
genetic_binner.fit(X[['青云24']], y)
print("【遗传算法分箱】")
print(genetic_binner.get_bin_table('青云24')[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【遗传算法分箱】
 分箱  样本总数     坏样本率
  0  9744 0.179392
  1  5402 0.149204
  2  4393 0.123833
  3  1378 0.113208
  4  1812 0.072296


In [8]:
# 平滑分箱 - 正则化
smooth_binner = OptimalBinning(method='smooth', max_n_bins=5)
smooth_binner.fit(X[['青云24']], y)
print("【平滑分箱】")
print(smooth_binner.get_bin_table('青云24')[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【平滑分箱】
 分箱  样本总数     坏样本率
  0 15146 0.168625
  1  7583 0.109587


In [9]:
# 核密度分箱 - 密度估计
kd_binner = OptimalBinning(method='kernel_density', max_n_bins=5)
kd_binner.fit(X[['青云24']], y)
print("【核密度分箱】")
print(kd_binner.get_bin_table('青云24')[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【核密度分箱】
 分箱  样本总数     坏样本率
  0 17300 0.162139
  1  5429 0.106834


In [10]:
# Best Lift分箱 - 提升度优化
lift_binner = OptimalBinning(method='best_lift', max_n_bins=5)
lift_binner.fit(X[['青云24']], y)
print("【Best Lift分箱】")
print(lift_binner.get_bin_table('青云24')[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【Best Lift分箱】
 分箱  样本总数     坏样本率
  0  1776 0.201577
  1  8169 0.175787
  2 10928 0.133327
  3  1856 0.072198


In [11]:
# 目标坏样本率分箱
tbr_binner = OptimalBinning(method='target_bad_rate', max_n_bins=5)
tbr_binner.fit(X[['青云24']], y)
print("【目标坏样本率分箱】")
print(tbr_binner.get_bin_table('青云24')[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【目标坏样本率分箱】
 分箱  样本总数     坏样本率
  0 20873 0.155751
  1   427 0.079625
  2   796 0.076633
  3   262 0.068702
  4   371 0.056604


In [12]:
# K-Means分箱 - 聚类算法
kmeans_binner = OptimalBinning(method='kmeans', max_n_bins=5)
kmeans_binner.fit(X[['青云24']], y)
print("【K-Means分箱】")
print(kmeans_binner.get_bin_table('青云24')[['分箱', '样本总数', '坏样本率']].to_string(index=False))

【K-Means分箱】
 分箱  样本总数     坏样本率
  0  2920 0.190068
  1  5655 0.174359
  2  6692 0.154812
  3  5256 0.119673
  4  2206 0.081142


## 5. 单调性约束(支持U型/倒U型)

In [13]:
# monotonic 参数支持多种模式
monotonic_modes = [
    ('ascending', '单调递增'),
    ('descending', '单调递减'),
    ('peak', '倒U型/峰值'),
    ('valley', 'U型/谷值'),
    ('auto', '自动检测'),
]

print("单调性约束模式对比(使用 monotonic 方法):")
print("="*60)
print(f"{'模式':<15} {'说明':<15} {'分箱数':>8} {'IV值':>10}")
print("-"*60)

feature = '游昆定制分80'
for mode, desc in monotonic_modes:
    binner = OptimalBinning(method='monotonic', monotonic=mode, max_n_bins=5)
    binner.fit(X[[feature]], y)
    bin_table = binner.get_bin_table(feature)
    n_bins = len([b for b in bin_table['分箱'] if b != 'missing'])
    iv = bin_table['分档IV值'].sum()
    print(f"{mode:<15} {desc:<15} {n_bins:>8} {iv:>10.4f}")
    
    if mode == 'auto':
        print(f"    检测到的模式: {binner.monotonic_trend_}")

单调性约束模式对比(使用 monotonic 方法):
模式              说明                   分箱数        IV值
------------------------------------------------------------
ascending       单调递增                   2     0.0706
descending      单调递减                   5     0.0881
peak            倒U型/峰值                 5     0.0881
valley          U型/谷值                  5     0.0881
auto            自动检测                   5     0.0881
    检测到的模式: {'游昆定制分80': 'descending'}


## 6. 预分箱功能

In [14]:
# 预分箱功能:类似optbinning,MDLP分箱前先做CART预分箱
# 支持传入分箱器、方法名或配置字典

feature = '百融定制分V9'

print(f"【{feature}】预分箱功能对比:")
print("="*70)

# 6.1 不使用预分箱
binner_no_pre = OptimalBinning(method='optimal_iv', max_n_bins=5)
binner_no_pre.fit(X[[feature]], y)
iv_no_pre = binner_no_pre.get_bin_table(feature)['分档IV值'].sum()
print(f"不使用预分箱: IV = {iv_no_pre:.4f}")

# 6.2 使用预分箱方法名
binner_pre_str = OptimalBinning(
    method='optimal_iv', 
    max_n_bins=5,
    prebinning='cart',
    prebinning_params={'max_n_bins': 20}
)
binner_pre_str.fit(X[[feature]], y)
iv_pre_str = binner_pre_str.get_bin_table(feature)['分档IV值'].sum()
print(f"使用cart预分箱(方法名): IV = {iv_pre_str:.4f}")

# 6.3 使用预分箱配置字典
binner_pre_dict = OptimalBinning(
    method='optimal_iv',
    max_n_bins=5, 
    prebinning={'method': 'cart', 'max_n_bins': 20}
)
binner_pre_dict.fit(X[[feature]], y)
iv_pre_dict = binner_pre_dict.get_bin_table(feature)['分档IV值'].sum()
print(f"使用cart预分箱(配置字典): IV = {iv_pre_dict:.4f}")

# 6.4 使用预分箱器实例
pre_binner = OptimalBinning(method='cart', max_n_bins=20)
binner_pre_obj = OptimalBinning(
    method='optimal_iv',
    max_n_bins=5,
    prebinning=pre_binner
)
binner_pre_obj.fit(X[[feature]], y)
iv_pre_obj = binner_pre_obj.get_bin_table(feature)['分档IV值'].sum()
print(f"使用cart预分箱(实例): IV = {iv_pre_obj:.4f}")

【百融定制分V9】预分箱功能对比:
不使用预分箱: IV = 0.1972
使用cart预分箱(方法名): IV = 0.2041
使用cart预分箱(配置字典): IV = 0.2041
使用cart预分箱(实例): IV = 0.2041


In [15]:
# MDLP分箱使用预分箱(参考optbinning实现)
print("MDLP分箱使用CART预分箱:")
print("="*60)

feature = '中智小牛分C3'

# 不使用预分箱
mdlp_no_pre = OptimalBinning(method='mdlp', max_n_bins=5)
mdlp_no_pre.fit(X[[feature]], y)
n_bins_no_pre = mdlp_no_pre.n_bins_[feature]
iv_no_pre = mdlp_no_pre.get_bin_table(feature)['分档IV值'].sum()
print(f"不使用预分箱: {n_bins_no_pre}箱, IV={iv_no_pre:.4f}")

# 使用CART预分箱
mdlp_with_pre = OptimalBinning(
    method='mdlp', 
    max_n_bins=5,
    prebinning='cart',
    prebinning_params={'max_n_bins': 20}
)
mdlp_with_pre.fit(X[[feature]], y)
n_bins_pre = mdlp_with_pre.n_bins_[feature]
iv_pre = mdlp_with_pre.get_bin_table(feature)['分档IV值'].sum()
print(f"使用CART预分箱: {n_bins_pre}箱, IV={iv_pre:.4f}")

MDLP分箱使用CART预分箱:
不使用预分箱: 4箱, IV=0.1195
使用CART预分箱: 5箱, IV=0.1136


## 7. 指定切分点

In [16]:
# 使用 user_splits 参数指定切分点
user_splits = {
    '青云24': [550, 600, 650, 700],
    '游昆定制分80': [600, 650, 700],
}

binner = OptimalBinning(user_splits=user_splits)
binner.fit(X, y)

print("指定切分点的分箱结果:")
print("="*60)
for feature in user_splits.keys():
    bin_table = binner.get_bin_table(feature)
    iv = bin_table['分档IV值'].sum()
    print(f"\n【{feature}】IV = {iv:.4f}")
    print(bin_table[['分箱', '样本总数', '坏样本率', '分档WOE值']].to_string(index=False))

指定切分点的分箱结果:

【青云24】IV = 0.0713
 分箱  样本总数     坏样本率    分档WOE值
  0  5238 0.190340  0.295225
  1  6096 0.164370  0.116963
  2  6068 0.134970 -0.114680
  3  3739 0.120353 -0.246063
  4  1588 0.073678 -0.788495

【游昆定制分80】IV = 0.0653
 分箱  样本总数     坏样本率    分档WOE值
  0    30 0.300000  0.895731
  1  5564 0.207405  0.402388
  2 17135 0.129676 -0.160797


## 8. WOE转换

In [17]:
# WOE转换用于逻辑回归等模型
binner = OptimalBinning(method='optimal_iv', max_n_bins=5)
binner.fit(X, y)

# WOE转换
X_woe = binner.transform(X, metric='woe')

print("WOE转换后的数据:")
print(X_woe.head())

print("\nWOE值范围:")
for col in X_woe.columns:
    print(f"{col}: [{X_woe[col].min():.2f}, {X_woe[col].max():.2f}]")

WOE转换后的数据:
       青云24   游昆定制分80   百融定制分V9   中智小牛分C3  度小满欺诈因子V6PRO多头版  百行百川分FPV1  游昆设备黑名单
0  0.050342 -0.307223 -0.854843 -0.424369         0.027574   0.046542      0.0
1 -0.233596 -0.307223 -0.854843 -0.922762         0.027574   0.046542      0.0
2  0.050342 -0.027107 -0.409758  0.046041         0.261297  -1.105689      0.0
3 -0.233596 -0.307223 -0.154689  0.046041        -0.382965  -1.105689      0.0
4 -0.233596 -0.307223 -0.154689  0.046041        -0.382965  -1.105689      0.0

WOE值范围:
青云24: [-0.81, 0.30]
游昆定制分80: [-0.31, 0.49]
百融定制分V9: [-0.85, 0.58]
中智小牛分C3: [-1.89, 0.40]
度小满欺诈因子V6PRO多头版: [-1.19, 0.56]
百行百川分FPV1: [-3.04, 0.66]
游昆设备黑名单: [0.00, 0.00]


## 9. 分箱标签

In [18]:
# 获取分箱标签(区间表示)
X_bins = binner.transform(X, metric='bins')

print("分箱标签:")
print(X_bins.head())

分箱标签:
               青云24         游昆定制分80               百融定制分V9          中智小牛分C3  \
0  (552.69, 632.11]   (716.0, +inf]      (847.2135, +inf]  (674.9, 752.15]   
1  (635.72, 693.48]   (716.0, +inf]      (847.2135, +inf]    (757.3, +inf]   
2  (552.69, 632.11]  (703.5, 716.0]  (807.2921, 847.2135]   (541.0, 674.9]   
3  (635.72, 693.48]   (716.0, +inf]  (761.5483, 807.2921]   (541.0, 674.9]   
4  (635.72, 693.48]   (716.0, +inf]  (761.5483, 807.2921]   (541.0, 674.9]   

      度小满欺诈因子V6PRO多头版         百行百川分FPV1       游昆设备黑名单  
0  (50.0856, 52.8292]  (149.04, 245.76]  (-inf, +inf)  
1  (50.0856, 52.8292]  (149.04, 245.76]  (-inf, +inf)  
2  (52.8292, 56.9395]   (271.8, 372.24]  (-inf, +inf)  
3  (43.2287, 50.0856]   (271.8, 372.24]  (-inf, +inf)  
4  (43.2287, 50.0856]   (271.8, 372.24]  (-inf, +inf)  


## 10. 特征选择

In [19]:
# 使用 OptimalBinning 进行特征筛选
binner = OptimalBinning(method='optimal_iv', max_n_bins=5)
binner.fit(X, y)

# 获取所有特征的IV值
print("特征IV值分析:")
print("="*60)
print(f"{'特征':<25} {'IV值':>10} {'预测能力':>12}")
print("-"*60)

iv_results = []
for feature in X.columns:
    bin_table = binner.get_bin_table(feature)
    iv = bin_table['分档IV值'].sum()
    iv_results.append((feature, iv))
    
    if iv < 0.02:
        level = '弱(剔除)'
    elif iv < 0.1:
        level = '中等'
    elif iv < 0.3:
        level = '强'
    else:
        level = '过强(检查)'
    
    print(f"{feature:<25} {iv:>10.4f} {level:>12}")

# 筛选建议保留的特征
keep_threshold = 0.02
selected_features = [f for f, iv in iv_results if iv >= keep_threshold]
print(f"\n建议保留的特征 (IV >= {keep_threshold}): {selected_features}")

特征IV值分析:
特征                               IV值         预测能力
------------------------------------------------------------
青云24                          0.0819           中等
游昆定制分80                       0.0927           中等
百融定制分V9                       0.1972            强
中智小牛分C3                       0.1297            强
度小满欺诈因子V6PRO多头版               0.0682           中等
百行百川分FPV1                     0.1497            强
游昆设备黑名单                       0.0000        弱(剔除)

建议保留的特征 (IV >= 0.02): ['青云24', '游昆定制分80', '百融定制分V9', '中智小牛分C3', '度小满欺诈因子V6PRO多头版', '百行百川分FPV1']


## 总结

`OptimalBinning` 统一接口的优势:

1. **一行代码切换方法**: 只需改 `method` 参数
2. **支持预分箱**: MDLP等方法可先预分箱再优化
3. **单调性约束**: 支持多种单调模式(U型/倒U型等)
4. **WOE转换**: 内置 WOE 编码功能
5. **指定切分点**: 支持自定义切分点

**推荐使用方式**:
```python
from hscredit.core.binning import OptimalBinning

# 基础使用
binner = OptimalBinning(method='optimal_iv', max_n_bins=5)
binner.fit(X_train, y_train)

# WOE转换
X_train_woe = binner.transform(X_train, metric='分档WOE值')
X_test_woe = binner.transform(X_test, metric='分档WOE值')
```